# 14 · Batch, CLI & plugins

The pieces above all build *one* figure. This notebook covers the **delivery** tier (RP.11) — turning the
same API into bulk output and shell/automation entry points:

| tool | what it does |
|---|---|
| `Batch` | render a whole series of inputs to image files in one call |
| `gallery` | embed those PNGs into one self-contained, static HTML page |
| `digitalearth` CLI | produce plots from a shell (`digitalearth plot` / `digitalearth batch`) |
| `load_plugins` | discover third-party styles/sources registered via entry points |

All of it is thin orchestration over `quickmap` — no new GIS or matplotlib machinery.

**Setup.** Resolve two bundled rasters from the repo root and a scratch output directory.

In [1]:
%matplotlib inline
import tempfile
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "acc4000.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DEM = str(ROOT / "examples" / "data" / "LisbonElevation.tif")
ACC = str(ROOT / "examples" / "data" / "acc4000.tif")
WORK = Path(tempfile.mkdtemp())
print("work dir:", WORK)

work dir: C:\Users\main\AppData\Local\Temp\tmpyfuxx3_0


## Batch — a series of inputs to image files

`Batch` takes shared plot options once, then renders each input (file path or in-memory pyramids object) to
`outdir`, closing every figure as it goes so a long run stays memory-bounded. It returns the written paths.

In [2]:
from digitalearth import Batch

batch = Batch(crs=3857, cmap="terrain", colorbar=True)
images = batch.run([DEM, ACC], WORK / "images")
[p.name for p in images]

['LisbonElevation.png', 'acc4000.png']

## gallery — one self-contained HTML page

`gallery` base64-embeds the PNGs into a single standalone `.html` (no server, no external assets) you can
open in any browser, email, or archive.

In [3]:
from digitalearth import gallery

page = gallery(
    images,
    WORK / "gallery.html",
    title="Digital-Earth batch gallery",
    captions=["Lisbon elevation", "Rhine flow accumulation"],
)
print(page, "—", page.stat().st_size, "bytes")

C:\Users\main\AppData\Local\Temp\tmpyfuxx3_0\gallery.html — 262320 bytes


## CLI — plots from a shell

Installed, the package exposes a `digitalearth` command (also `python -m digitalearth`). The cell below calls
the same entry point in-process; it is equivalent to running:

```bash
digitalearth plot acc4000.tif -o map.png --crs 3857
digitalearth batch *.tif -o images/ --html gallery.html --crs 3857
```

`main` returns a process exit code (`0` on success); progress is logged to stderr.

In [4]:
from digitalearth.cli import main

main(["plot", ACC, "-o", str(WORK / "map.png"), "--crs", "3857"])

wrote C:\Users\main\AppData\Local\Temp\tmpyfuxx3_0\map.png


0

## plugins — third-party extensions

Other packages can register extra styles or source adapters under the `digitalearth.*` entry-point groups
(declared in *their* `pyproject.toml`). `load_plugins` discovers and loads them. Here we simulate one by
passing an entry point directly instead of installing a package.

In [5]:
from digitalearth import load_plugins

class _StylePlugin:
    name = "tropical"
    def load(self):
        return {"cmap": "turbo", "levels": 12}

load_plugins("digitalearth.styles", eps=[_StylePlugin()])

{'tropical': {'cmap': 'turbo', 'levels': 12}}